In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Customize high-resolution display for charts
%config InlineBackend.figure_format = 'retina'

# 1. Find analysis files
csv_files = glob.glob("report_metrics_*.csv")
if not csv_files:
    print("No CSV files found starting with 'report_metrics_'")
else:
    print(f"Found {len(csv_files)} files: {csv_files}")

    # 2. Merge data
    dfs = [pd.read_csv(f) for f in csv_files]
    df_all = pd.concat(dfs, ignore_index=True)

    # 3. Preprocessing
    if df_all['Accuracy'].max() <= 1.0:
        df_all['Accuracy'] = df_all['Accuracy'] * 100

    num_rounds_per_task = df_all['Round'].max()
    df_all['Global_Round'] = df_all['Train_Task'] * num_rounds_per_task + df_all['Round']

    print(f"Loaded {df_all.shape[0]} rows of data. Generating charts...")

    # 4. Plot charts
    sns.set_theme(style="whitegrid")
    eval_tasks = sorted(df_all['Eval_Task'].unique())
    num_eval_tasks = len(eval_tasks)

    fig, axes = plt.subplots(nrows=num_eval_tasks, ncols=1, 
                             figsize=(12, 5 * num_eval_tasks), 
                             sharex=True)

    if num_eval_tasks == 1:
        axes = [axes]

    for ax, eval_id in zip(axes, eval_tasks):
        subset = df_all[df_all['Eval_Task'] == eval_id]
        
        sns.lineplot(
            data=subset,
            x='Global_Round',
            y='Accuracy',
            hue='Method',
            linewidth=2.5,
            ax=ax,
            palette="tab10"
        )
        
        ax.set_title(f"Accuracy on Test Set of Task {eval_id}", fontweight='bold', fontsize=14)
        ax.set_ylabel("Accuracy (%)", fontsize=12)
        ax.set_ylim(-5, 105)

        # Draw separator lines for Tasks
        for t_split in df_all['Train_Task'].unique():
            if t_split > 0:
                split_round = t_split * num_rounds_per_task
                ax.axvline(x=split_round, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
                ax.text(split_round + 1, 5, f'Start Task {t_split}', color='red', fontsize=10, rotation=90)
                
        ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', title="Strategies")

    axes[-1].set_xlabel("Global Rounds", fontsize=12)
    plt.tight_layout()

    # Display the chart directly in the notebook
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

# ==========================================
# 1. ORIGINAL PAPER BASELINES (Hardcoded)
# ==========================================
PAPER_BASELINES = {
    'CIFAR10': {
        0.5: {'Finetune': 38.71, '+GDR': 62.13, '+TTS': 41.34, '+GDR+TTS': 64.11},
        1.0: {'Finetune': 40.49, '+GDR': 63.81, '+TTS': 42.55, '+GDR+TTS': 65.20}
    },
    'CIFAR100': {
        0.1: {'Finetune': 15.17, '+GDR': 45.28, '+TTS': 17.32, '+GDR+TTS': 46.40},
        0.5: {'Finetune': 16.75, '+GDR': 47.66, '+TTS': 17.14, '+GDR+TTS': 49.76},
        1.0: {'Finetune': 17.15, '+GDR': 51.47, '+TTS': 19.32, '+GDR+TTS': 52.06}
    },
    'TinyImageNet': {
        0.1: {'Finetune': 6.06, '+GDR': 17.24, '+TTS': 6.67, '+GDR+TTS': 18.37},
        0.5: {'Finetune': 6.00, '+GDR': 17.89, '+TTS': 6.92, '+GDR+TTS': 18.86},
        1.0: {'Finetune': 6.40, '+GDR': 18.04, '+TTS': 7.27, '+GDR+TTS': 18.78}
    }
}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def extract_final_average_accuracy(csv_path):
    """Reads the CSV log and returns the average accuracy of the final round."""
    if not os.path.exists(csv_path):
        print(f"  [!] Warning: File not found -> {csv_path}. Returning 0.0")
        return 0.0
    
    try:
        df = pd.read_csv(csv_path)
        final_train_task = df['Train_Task'].max()
        final_round = df[df['Train_Task'] == final_train_task]['Round'].max()
        final_results = df[(df['Train_Task'] == final_train_task) & (df['Round'] == final_round)]
        avg_acc = final_results['Accuracy'].mean()
        return round(avg_acc, 2)
    except Exception as e:
        print(f"  [!] Error reading {csv_path}: {e}")
        return 0.0

def get_experiment_folder(base_dir, dataset_name, beta_value):
    """Automatically finds the correct folder for the given dataset and beta."""
    # Convert dataset name to lowercase to match folder names (e.g., CIFAR10 -> cifar10)
    dataset_folder = os.path.join(base_dir, dataset_name.lower())
    if not os.path.exists(dataset_folder):
        print(f"Dataset directory not found: {dataset_folder}")
        return None
    
    # Search for a folder that ends with the exact beta value
    search_pattern = os.path.join(dataset_folder, f"*-{beta_value}")
    matching_folders = glob.glob(search_pattern)
    
    # Filter only directories
    matching_folders = [f for f in matching_folders if os.path.isdir(f)]
    
    if matching_folders:
        return matching_folders[0]  # Return the first matching directory
    else:
        print(f"Experiment folder not found for Beta={beta_value} inside {dataset_folder}")
        return None

def plot_reproduction_comparison(dataset_name, beta_value, base_dir="report"):
    """Plots a grouped bar chart dynamically reading from the directory structure."""
    methods = ['Finetune', '+GDR', '+TTS', '+GDR+TTS']
    
    # Standardized file names based on your consistency fix
    file_names = {
        'Finetune': 'report_metrics_none.csv',
        '+GDR': 'report_metrics_gdr.csv',
        '+TTS': 'report_metrics_random_tts.csv',
        '+GDR+TTS': 'report_metrics_gdr_tts.csv'
    }
    
    # 1. Fetch Paper Baseline Data
    try:
        paper_dict = PAPER_BASELINES[dataset_name][beta_value]
        paper_scores = [paper_dict[m] for m in methods]
    except KeyError:
        print(f"Error: No paper baseline data found for {dataset_name} (Beta={beta_value})")
        return

    # 2. Find the correct directory
    exp_folder = get_experiment_folder(base_dir, dataset_name, beta_value)
    if not exp_folder:
        return

    print(f"Reading data from: {exp_folder}")

    # 3. Read Data
    repro_scores = []
    for method in methods:
        csv_file_path = os.path.join(exp_folder, file_names[method])
        acc = extract_final_average_accuracy(csv_file_path)
        repro_scores.append(acc)

    # 4. Plotting Setup
    x = np.arange(len(methods))
    width = 0.35  
    
    fig, ax = plt.subplots(figsize=(10, 6))
    rects1 = ax.bar(x - width/2, paper_scores, width, label='Original Paper', color='#4C72B0')
    rects2 = ax.bar(x + width/2, repro_scores, width, label='Our Reproduction', color='#DD8452')
    
    # Customization
    ax.set_ylabel('Average Accuracy (%)', fontsize=12, fontweight='bold')
    ax.set_title(f'Performance Comparison: Paper vs Reproduction\nDataset: {dataset_name} | Dirichlet $\\beta$ = {beta_value}', 
                 fontsize=14, fontweight='bold', pad=15)
    ax.set_xticks(x)
    ax.set_xticklabels(methods, fontsize=12)
    ax.set_ylim(0, max(max(paper_scores), max(repro_scores)) + 15) # Add space for text
    
    ax.legend(fontsize=11, loc='upper left')
    ax.yaxis.grid(True, linestyle='--', alpha=0.7)
    ax.set_axisbelow(True)
    
    # Value Annotations
    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height}%',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), 
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    autolabel(rects1)
    autolabel(rects2)
    
    plt.tight_layout()
    plt.show()

# ==========================================
# 3. EXECUTION BLOCK
# ==========================================
# Make sure your notebook is in the same folder that contains the 'report/' directory.

# Choose your dataset and beta here:
TARGET_DATASET = 'CIFAR10'   # Must exactly match the keys in PAPER_BASELINES (e.g., 'CIFAR10', 'CIFAR100')
TARGET_BETA = 0.5            # 0.1, 0.5, or 1.0

# plot_reproduction_comparison(
#     dataset_name=TARGET_DATASET, 
#     beta_value=TARGET_BETA, 
#     base_dir="report"        # The root folder of your logs
# )

# You can easily loop through everything you have:
for b in [0.1, 0.5, 1.0]:
   plot_reproduction_comparison('TinyImageNet', b)